In [2]:
import sys
import os

sys.path.append(os.path.realpath(os.path.join(os.getcwd(), '..', '..', '..', 'T5000', 'T5000-Base', 'Pylians3', 'library')))
sys.path.append(os.path.abspath('../src'))

from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import networkx as nx

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils import load_config
from src.validation import get_tpcf
from src.dataset import deterministic_sample

import MAS_library as MASL
import Pk_library as PKL

In [9]:
subbox_halos = np.load("../../Data/subboxes/train_subbox_halos.npy")
subbox_counts = np.load("../../Data/subboxes/train_subbox_counts.npy")

# for r in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
# for _ in range(100):
degrees = []
diameters = []
r = 0.3
    
for idx in range(2000):
    n = subbox_counts[idx]
    x = torch.tensor(subbox_halos[idx, :n, :3] / 370.0, dtype=torch.float32)
    
    diff = x.unsqueeze(0) - x.unsqueeze(1)
    diff = diff - torch.round(diff)
    dist = diff.norm(dim=-1)
    
    adj = (dist < r) & (dist > 0)
    degrees.append(adj.sum(dim=1).float().mean().item())
    
    G = nx.from_numpy_array(adj.numpy())
    if nx.is_connected(G):
        pass
    else:
        print(float('inf'))

# print(f"r={r:.2f} | avg degree: {np.mean(degrees):.1f}, "
#       f"avg diameter: {np.mean(diameters):.1f}, "
#       f"disconnected: {sum(d == float('inf') for d in diameters)}/20")